## Libraries

In [14]:
import os
import sys
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import config
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

## Config

In [2]:
data_dir = config.DATA_DIR
train_folder_path = config.TRAIN_DATA_DIR
test_folder_path = config.TEST_DATA_DIR
image_extensions = {".jpg"}

## Read Dataset

In [3]:
asl_alphabet = pd.read_csv(os.path.join(train_folder_path, "landmarks.csv"))
print(asl_alphabet.shape)
asl_alphabet.head()

(52419, 89)


,image_path,label,landmark_00_x,landmark_00_y,landmark_00_z,landmark_01_x,landmark_01_y,landmark_01_z,landmark_02_x,landmark_02_y,...,pip_angle_thumb,pip_angle_index,pip_angle_middle,pip_angle_ring,pip_angle_pinky,extension_ratio_thumb,extension_ratio_index,extension_ratio_middle,extension_ratio_ring,extension_ratio_pinky
0,/Users/gabrielvictorgomesferreira/artificial_i...,A,0.0,0.0,0.0,0.373646,-0.271498,-0.117752,0.601571,-0.700774,...,2.756618,1.273215,1.272128,1.457126,1.453901,0.962049,0.603282,0.611880,0.660277,0.650234
1,/Users/gabrielvictorgomesferreira/artificial_i...,A,0.0,0.0,0.0,0.366587,-0.236147,-0.090191,0.589979,-0.655155,...,2.756422,1.267653,1.456597,1.673530,1.768365,0.971620,0.606007,0.688835,0.737631,0.755499
2,/Users/gabrielvictorgomesferreira/artificial_i...,A,0.0,0.0,0.0,0.336885,-0.249354,-0.137063,0.580642,-0.734355,...,2.848757,0.929390,0.811522,0.729440,0.512162,0.975596,0.444591,0.432206,0.472180,0.405715
3,/Users/gabrielvictorgomesferreira/artificial_i...,A,0.0,0.0,0.0,0.335918,-0.286065,-0.122139,0.522601,-0.726767,...,2.729671,1.278139,1.206833,1.242217,1.401074,0.974153,0.580692,0.580509,0.592699,0.625437
4,/Users/gabrielvictorgomesferreira/artificial_i...,A,0.0,0.0,0.0,0.339923,-0.265747,-0.119792,0.540812,-0.735016,...,2.710071,1.277354,1.222245,1.243615,1.426648,0.973767,0.589968,0.590129,0.603073,0.643955


In [4]:
# Encode string labels to integers
labels = asl_alphabet["label"].values
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)
feature_df = asl_alphabet.drop(columns=["label", "image_path"], errors="ignore")
feature_matrix = feature_df.values.astype(np.float32)
print(f"\nLabel encoder classes: {list(label_encoder.classes_)}")


Label encoder classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']


## Split Data into Train and Test

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
        feature_matrix,
        encoded_labels,
        test_size=.2,
        stratify=encoded_labels,
        random_state=42
    )
print(f"Training set: {len(X_train)} samples")
print(f"Test set:     {len(X_test)} samples")

Training set: 41935 samples
Test set:     10484 samples


In [6]:
# y_train

## Model Training
### Random Florest

In [7]:
rf_model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1  
    )
rf_model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

### SVM Model

In [8]:
svm_scaler = StandardScaler()
X_train_scaled = svm_scaler.fit_transform(X_train)

svm_model = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)
svm_model.fit(X_train_scaled, y_train)

SVC(probability=True, random_state=42)

## Model Testing

In [9]:
def evaluate_model(model, X_test: np.ndarray, y_test: np.ndarray, model_name: str, scaler=None):
    # Apply scaling if needed
    if scaler is not None:
        X_test_scaled = scaler.transform(X_test)
    else:
        X_test_scaled = X_test

    predictions = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, predictions)

    print(f"\n{'='*50}")
    print(f"Results for: {model_name}")
    print(f"{'='*50}")
    print(f"Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print()
    print(classification_report(y_test, predictions))

    return accuracy

In [10]:
rf_accuracy = evaluate_model(rf_model, X_test, y_test, "Random Forest", scaler=None)
svm_accuracy = evaluate_model(svm_model, X_test, y_test, "SVM (RBF kernel)", scaler=svm_scaler)


Results for: Random Forest
Overall Accuracy: 0.9927 (99.27%)

              precision    recall  f1-score   support

           0       1.00      0.99      1.00       417
           1       1.00      1.00      1.00       430
           2       0.99      1.00      0.99       356
           3       1.00      0.99      0.99       460
           4       0.99      0.99      0.99       440
           5       0.99      1.00      1.00       553
           6       1.00      1.00      1.00       458
           7       1.00      1.00      1.00       459
           8       1.00      0.99      1.00       454
           9       1.00      0.99      1.00       520
          10       1.00      1.00      1.00       484
          11       0.92      0.97      0.94       235
          12       0.95      0.92      0.94       197
          13       0.99      1.00      0.99       431
          14       0.99      0.99      0.99       385
          15       1.00      0.98      0.99       390
          16      

## Model Selection

In [11]:
model_results = [
    ("Random Forest", rf_model, None, rf_accuracy),
    ("SVM", svm_model, svm_scaler, svm_accuracy),
]

# Sort by accuracy descending
model_results.sort(key=lambda x: x[3], reverse=True)
best_name, best_model, best_scaler, best_accuracy = model_results[0]

print("\n" + "=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
for rank, (name, _, _, accuracy) in enumerate(model_results, start=1):
    marker = " <-- BEST" if rank == 1 else ""
    print(f"  #{rank} {name}: {accuracy*100:.2f}%{marker}")

second_best_accuracy = model_results[1][3]
accuracy_gap = best_accuracy - second_best_accuracy
print(f"\nWinner: {best_name} with {best_accuracy*100:.2f}% accuracy")
print(f"Margin over #2: {accuracy_gap*100:.2f} percentage points")


MODEL COMPARISON SUMMARY
  #1 SVM: 99.28% <-- BEST
  #2 Random Forest: 99.27%

Winner: SVM with 99.28% accuracy
Margin over #2: 0.01 percentage points


## Save Model

In [12]:
def save_model_artifacts(model, scaler, label_encoder, model_name: str):
    os.makedirs(config.MODELS_DIR, exist_ok=True)

    print(f"\nSaving {model_name} to: {config.BEST_MODEL_PATH}")
    joblib.dump(model, config.BEST_MODEL_PATH)

    if scaler is not None:
        print(f"Saving scaler to: {config.SCALER_PATH}")
        joblib.dump(scaler, config.SCALER_PATH)
    else:
        print("No scaler to save (Random Forest doesn't need one).")

    print(f"Saving label encoder to: {config.LABEL_ENCODER_PATH}")
    joblib.dump(label_encoder, config.LABEL_ENCODER_PATH)

    print("Models saved successfully.")

In [15]:
# Save the best model
save_model_artifacts(best_model, best_scaler, label_encoder, best_name)


Saving SVM to: /Users/gabrielvictorgomesferreira/artificial_intelligence/isu_classes/projects/ASL-English-Fingerspelling/models/best_model.pkl
Saving scaler to: /Users/gabrielvictorgomesferreira/artificial_intelligence/isu_classes/projects/ASL-English-Fingerspelling/models/scaler.joblib
Saving label encoder to: /Users/gabrielvictorgomesferreira/artificial_intelligence/isu_classes/projects/ASL-English-Fingerspelling/models/label_encoder.joblib
Models saved successfully.
